# Деплой Arena-ветки на Cloudflare **без мержа в main**

Живой бот сейчас с **ручного деплоя `main`** (5 дней назад) — поэтому `/diag` нет.

Этот ноутбук:
1. ставит Node.js;
2. клонирует **не `main`**, а рабочую ветку `arena/01a020b8-threadsbot`;
3. делает `wrangler deploy` на тот же Worker `threadsbot`;
4. **не пушит и не мержит** в `main`.

GitHub `main` не меняется. Меняется только то, что крутится на Cloudflare.

Открыть в Colab после пуша в ветку:
https://colab.research.google.com/github/Bergaff/threadsbot/blob/arena/01a020b8-threadsbot/notebooks/deploy_branch_no_merge.ipynb

Нужно: **Workers Paid** (уже куплен), Cloudflare API Token с правом Workers Edit, Account ID.



## Ячейка 1 — Node.js 20

Colab без Node не соберёт Worker. Ставим в `/usr/local`, один раз на сессию.



In [ ]:
import shutil, subprocess, os, sys

def sh(cmd, **kw):
    print('+', cmd if isinstance(cmd, str) else ' '.join(cmd))
    subprocess.run(cmd, check=True, **kw)

if shutil.which('node'):
    print('node', subprocess.check_output(['node','-v'], text=True).strip())
else:
    sh('curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.tar.xz', shell=True)
    sh('tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1', shell=True)
    print('node', subprocess.check_output(['node','-v'], text=True).strip())
print('npm', subprocess.check_output(['npm','-v'], text=True).strip())



## Ячейка 2 — какая ветка и куда деплоим

`BRANCH` — Arena, не `main`. Менять `main` не нужно и нельзя для этой задачи.



In [ ]:
REPO = 'https://github.com/Bergaff/threadsbot.git'
BRANCH = 'arena/01a020b8-threadsbot'   # не main
PROJECT = '/content/threadsbot'
WORKER_NAME = 'threadsbot'
print('Будем деплоить ветку:', BRANCH)
print('Репозиторий:', REPO)
print('GitHub main НЕ трогаем')



## Ячейка 3 — токены Cloudflare (скрытый ввод)

Создать токен: https://dash.cloudflare.com/profile/api-tokens  
Шаблон **Edit Cloudflare Workers** (Account.Cloudflare Workers:Edit, Account.Account Settings:Read, Zone если спросит).

Account ID: Workers & Pages справа, или URL `dash.cloudflare.com/<ID>/...`



In [ ]:
import getpass, os

cf_token = getpass.getpass('Cloudflare API Token (скрыто): ').strip()
cf_account = getpass.getpass('Cloudflare Account ID (скрыто): ').strip()
if not cf_token or not cf_account:
    raise ValueError('Нужны оба поля')
os.environ['CLOUDFLARE_API_TOKEN'] = cf_token
os.environ['CLOUDFLARE_ACCOUNT_ID'] = cf_account
print('Токен принят, в вывод не печатаю. Длина токена:', len(cf_token))



## Ячейка 4 — клонировать **рабочую ветку**, не main

`--branch` сразу берёт Arena. Если папка уже есть — `fetch` + `checkout`.



In [ ]:
import os, subprocess, shutil
from pathlib import Path

def sh(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd)

if Path(PROJECT).exists() and (Path(PROJECT)/'.git').exists():
    sh(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT)
    sh(['git', 'checkout', BRANCH], cwd=PROJECT)
    sh(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=PROJECT)
else:
    if Path(PROJECT).exists():
        shutil.rmtree(PROJECT)
    sh(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, PROJECT])

print('\n--- git ---')
subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=PROJECT, check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=PROJECT, check=True)
branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=PROJECT, text=True).strip()
if branch == 'main':
    raise RuntimeError('Сейчас main — СТОП. Нужна ветка ' + BRANCH)
print('✅ На ветке', branch, '— можно деплоить')



## Ячейка 5 — npm install

Только зависимости Worker. Python-бот не трогаем.



In [ ]:
import subprocess
subprocess.run(['npm', 'install'], cwd=PROJECT, check=True)
print('✅ npm install')



## Ячейка 6 — кто мы в Cloudflare + очередь

Queue `threadsbot-updates` обязательна для кнопок Текст/Скрины. Если её нет — создаём. `main` в git не меняется.



In [ ]:
import subprocess, json

def run(args):
    print('+', ' '.join(args))
    return subprocess.run(args, cwd=PROJECT, check=True, text=True, capture_output=True)

who = subprocess.run(['npx', 'wrangler', 'whoami'], cwd=PROJECT, check=False, text=True, capture_output=True)
print(who.stdout or who.stderr)
if who.returncode != 0:
    raise RuntimeError('wrangler whoami не прошёл — проверь токен (Edit Cloudflare Workers)')

q = subprocess.run(['npx', 'wrangler', 'queues', 'list'], cwd=PROJECT, text=True, capture_output=True)
print(q.stdout or q.stderr)
if 'threadsbot-updates' not in (q.stdout or '') + (q.stderr or ''):
    print('Очереди нет — создаю threadsbot-updates')
    subprocess.run(['npx', 'wrangler', 'queues', 'create', 'threadsbot-updates'], cwd=PROJECT, check=True)
else:
    print('✅ очередь threadsbot-updates есть')



## Ячейка 7 — **деплой Worker с этой ветки**

Это и есть «смена ветки» для Cloudflare: на Worker уезжает код Arena, GitHub `main` остаётся старым.

Секреты (`TELEGRAM_TOKEN` и т.д.) wrangler не затирает.



In [ ]:
import subprocess, re

proc = subprocess.run(['npx', 'wrangler', 'deploy'], cwd=PROJECT, text=True, capture_output=True)
out = (proc.stdout or '') + '\n' + (proc.stderr or '')
print(out)
if proc.returncode != 0:
    raise RuntimeError('deploy упал — смотри вывод выше')
m = re.search(r'https://[\w.-]+\.workers\.dev', out)
WORKER_URL = m.group(0).rstrip('/') if m else ''
print('\n✅ Deploy OK. URL:', WORKER_URL or '(не распарсил, возьми из вывода)')



## Ячейка 8 — проверка `/health`

Живой новый код: `"version": "pr5-2026-08-20-deploy"`.  
Если `unknown` или нет поля `version` — задеплоен всё ещё старый main.



In [ ]:
import urllib.request, json, re

url = WORKER_URL if 'WORKER_URL' in globals() and WORKER_URL else input('URL Worker (https://threadsbot.XXX.workers.dev): ').strip().rstrip('/')
print('GET', url + '/health')
with urllib.request.urlopen(url + '/health', timeout=30) as resp:
    health = json.loads(resp.read().decode())
print(json.dumps(health, ensure_ascii=False, indent=2))
ver = str(health.get('version', 'unknown'))
if ver != 'pr5-2026-08-20-deploy':
    print('\n⚠️ version не тот. Ожидали pr5-2026-08-20-deploy, получили', ver)
    print('Значит на проде ещё старый main, либо деплой ушёл в другой Worker.')
else:
    print('\n✅ Это Arena-код. В боте теперь есть /diag')
if not health.get('queue'):
    print('⚠️ queue=false — кнопки Текст/Скрины не заработают')
if not health.get('browser'):
    print('⚠️ browser=false — нет Browser Rendering binding')
accounts = health.get('accounts') or {}
print('аккаунты alive/total:', accounts.get('alive'), '/', accounts.get('total'))



## Ячейка 9 — необязательно: webhook

Если бот не отвечает после деплоя — URL Worker мог смениться, webhook надо прописать снова.

`WEBHOOK_SECRET` тот же, что уже лежит в секретах Worker (который ставили при первом деплое).



In [ ]:
import getpass, urllib.request, json

do_wh = input('Прописать webhook заново? y/N: ').strip().lower()
if do_wh == 'y':
    url = WORKER_URL if 'WORKER_URL' in globals() and WORKER_URL else input('URL Worker: ').strip().rstrip('/')
    secret = getpass.getpass('WEBHOOK_SECRET (скрыто): ').strip()
    req = urllib.request.Request(
        url + '/setup-webhook',
        data=b'',
        headers={'Authorization': 'Bearer ' + secret},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        print(resp.status, resp.read().decode()[:500])
    print('✅ webhook вызван')
else:
    print('Пропущено. Если бот молчит — запусти ячейку ещё раз и ответь y.')



## Ячейка 10 — очистка токенов из Colab



In [ ]:
import os
for k in ('CLOUDFLARE_API_TOKEN', 'CLOUDFLARE_ACCOUNT_ID'):
    os.environ.pop(k, None)
cf_token = cf_account = ''
print('Токены стёрты из окружения. Runtime Colab после работы: Runtime → Disconnect and delete runtime.')



## После деплоя

1. В Telegram боту: `/diag` — должен ответить `version: pr5-2026-08-20-deploy`.
2. `/accounts` — если пусто, пришли JSON.
3. `zuck` → Текст. Paid уже есть, первый запрос 30–60 сек.
4. GitHub `main` не менялся. PR https://github.com/Bergaff/threadsbot/pull/5 по-прежнему не смержен.
5. Если коллега genver135 снова сделает Manual deploy с `main` — бот откатится. Деплой только с этой ветки, пока проверяем Arena.

